In [ ]:
import gcamreader
from lxml import etree
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH    = '../output'
DB_FILE    = 'database_basexdb'
QUERY_FILE = '../output/queries/Main_queries.xml'

SSP_COLORS = {
    'GCAM_SSP1': '#2ecc71',
    'GCAM_SSP2': '#3498db',
    'GCAM_SSP3': '#e74c3c',
    'GCAM_SSP4': '#f39c12',
    'GCAM_SSP5': '#9b59b6',
}

MTC_TO_GTCO2 = (44 / 12) / 1000

conn = gcamreader.querymi.LocalDBConn(DB_PATH, DB_FILE)
query_tree = etree.parse(QUERY_FILE)

available = conn.listScenariosInDB()['name'].tolist()
print('Scenarios in DB:', available)

In [ ]:
def get_query(tree, title, tag='emissionsQueryBuilder'):
    for elem in tree.getroot().iter(tag):
        if elem.get('title') == title:
            return gcamreader.querymi.Query(elem)
    raise KeyError(f'Query not found: {title}')

def query_global_co2(scenarios):
    q = get_query(query_tree, 'CO2 emissions by region')
    df = conn.runQuery(q, scenarios=scenarios)
    return (
        df.groupby(['scenario', 'Year'], as_index=False)['value']
        .sum()
        .assign(GtCO2=lambda d: d['value'] * MTC_TO_GTCO2)
        .drop(columns='value')
    )

# Pull all available scenarios
all_scenarios = [s for s in ['Reference'] + list(SSP_COLORS.keys()) if s in available]
df = query_global_co2(all_scenarios)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for scenario, grp in df.groupby('scenario'):
    label = scenario.replace('GCAM_', '')
    if scenario == 'Reference':
        ax.plot(grp['Year'], grp['GtCO2'], color='#aaaaaa', linewidth=2, label='Reference')
    else:
        ax.plot(grp['Year'], grp['GtCO2'], color=SSP_COLORS[scenario], linewidth=2, label=label)

ax.set_xlabel('Year')
ax.set_ylabel('GtCO₂ / yr')
ax.set_ylim(0)
ax.legend(frameon=False)
ax.grid(axis='y', linewidth=0.4, alpha=0.4)

plt.tight_layout()
plt.savefig('../output/gcam_diagnostics/co2_total.png', dpi=150)
plt.show()